In [1]:
!pip install transformers datasets sentencepiece

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments, Trainer

# STEP 3: Load dataset (BillSum -> legal text + summaries)
dataset = load_dataset("billsum", split="train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/91.8M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

data/ca_test-00000-of-00001.parquet:   0%|          | 0.00/6.12M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18949 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3269 [00:00<?, ? examples/s]

Generating ca_test split:   0%|          | 0/1237 [00:00<?, ? examples/s]

In [3]:
# STEP 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set base directory in Google Drive where model will be saved
drive_base = "/content/drive/MyDrive/legal_simplifier"

Mounted at /content/drive


In [20]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq

# Tokenizer
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Preprocessing function
def preprocess(example):
    # Flatten text and summary lists into strings
    input_text = " ".join(example["text"]) if isinstance(example["text"], list) else example["text"]
    target_text = " ".join(example["summary"]) if isinstance(example["summary"], list) else example["summary"]

    input_text = "simplify: " + input_text

    # Tokenize input (pad here)
    model_inputs = tokenizer(
        input_text,
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    # Tokenize labels (no padding)
    labels = tokenizer(
        target_text,
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Map the preprocessing function one example at a time
tokenized_dataset = dataset.map(preprocess, batched=False, remove_columns=dataset.column_names)

# Data collator for dynamic padding
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/legal_simplifier/checkpoints",
    save_strategy="epoch",
    save_total_limit=3,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    logging_dir="/content/drive/MyDrive/legal_simplifier/logs",
    logging_steps=50
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset.select(range(100)),  # small validation
    tokenizer=tokenizer,
    data_collator=data_collator
)

# Train
trainer.train()

# Save final model
final_model_dir = "/content/drive/MyDrive/legal_simplifier/final_model"
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
print(f"✅ Model saved at {final_model_dir}")


Map:   0%|          | 0/18949 [00:00<?, ? examples/s]

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-443445600.py:52: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
50,3.471800
100,2.899300
150,2.708200
200,2.612100
250,2.632100
300,2.477300
350,2.474400
400,2.446600
450,2.557000
500,2.537900


✅ Model saved at /content/drive/MyDrive/legal_simplifier/final_model


In [21]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

final_model_dir = "/content/drive/MyDrive/legal_simplifier/final_model"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(final_model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(final_model_dir)


In [22]:
legal_text = """
The party of the first part shall not be liable for any indirect, incidental, or consequential damages arising out of or in connection with this agreement.
"""
input_text = "simplify: " + legal_text  # add the task prefix if your model was trained with it


In [23]:
inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)

# Generate output
output_ids = model.generate(
    **inputs,
    max_length=150,
    num_beams=4,      # beam search for better output
    early_stopping=True
)

# Decode
simplified_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("🔹 Simplified Text:\n", simplified_text)


🔹 Simplified Text:
 Directs the parties of the first part to not be liable for any indirect, incidental, or consequential damages arising out of or in connection with this agreement.
